# 01 — Data Cleaning
**Author:** Nirwan Maharjan  

This notebook handles all preprocessing of the raw Audible dataset.  
Output: `../data/audible_cleaned.csv` — used by all subsequent notebooks.

### Steps
1. Load & inspect raw data
2. Fix `author` and `narrator` columns
3. Fix `price` column
4. Parse `time` → `duration_minutes`
5. Split `stars` → `star_score` + `num_ratings`
6. Parse `releasedate` → extract `release_year`
7. Fix `language` formatting
8. Remove duplicates
9. Final check & export

---
## 1. Imports & Load Raw Data

In [1]:
import pandas as pd
import numpy as np
import re
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('../data/audible_uncleaned.csv')

print(f'Shape: {df.shape}')
print(f'\nColumn dtypes:')
print(df.dtypes)
df.head()

Shape: (87489, 8)

Column dtypes:
name           str
author         str
narrator       str
time           str
releasedate    str
language       str
stars          str
price          str
dtype: object


,name,author,narrator,time,releasedate,language,stars,price
0,Geronimo Stilton #11 & #12,Writtenby:GeronimoStilton,Narratedby:BillLobely,2 hrs and 20 mins,04-08-08,English,5 out of 5 stars34 ratings,468.00
1,The Burning Maze,Writtenby:RickRiordan,Narratedby:RobbieDaymond,13 hrs and 8 mins,01-05-18,English,4.5 out of 5 stars41 ratings,820.00
2,The Deep End,Writtenby:JeffKinney,Narratedby:DanRussell,2 hrs and 3 mins,06-11-20,English,4.5 out of 5 stars38 ratings,410.00
3,Daughter of the Deep,Writtenby:RickRiordan,Narratedby:SoneelaNankani,11 hrs and 16 mins,05-10-21,English,4.5 out of 5 stars12 ratings,615.00
4,"The Lightning Thief: Percy Jackson, Book 1",Writtenby:RickRiordan,Narratedby:JesseBernstein,10 hrs,13-01-10,English,4.5 out of 5 stars181 ratings,820.00


In [2]:
# Check for nulls in raw data
print('Null counts per column:')
print(df.isnull().sum())

Null counts per column:
name           0
author         0
narrator       0
time           0
releasedate    0
language       0
stars          0
price          0
dtype: int64


In [3]:
# Preview each column's unique value patterns
for col in df.columns:
    print(f'\n--- {col} ---')
    print(df[col].value_counts().head(5))


--- name ---
name
The Art of War          20
Sterling Biographies    19
Sterling Point Books    16
The Odyssey             16
Hamlet                  15
Name: count, dtype: int64

--- author ---
author
Writtenby:矢島雅弘,石橋遊                   874
Writtenby:SmartReading               405
Writtenby:中西貴之,BJ                    311
Writtenby:div.                       274
Writtenby:OnlineStudioProductions    212
Name: count, dtype: int64

--- narrator ---
narrator
Narratedby:anonymous     1034
Narratedby:矢島雅弘,石橋遊       874
Narratedby:Intuitive      465
Narratedby:uncredited     326
Narratedby:中西貴之,BJ        311
Name: count, dtype: int64

--- time ---
time
2 mins    372
6 mins    369
5 mins    369
7 mins    356
4 mins    347
Name: count, dtype: int64

--- releasedate ---
releasedate
16-05-18    773
04-01-22    505
01-02-22    403
15-03-22    367
08-02-22    358
Name: count, dtype: int64

--- language ---
language
English     61884
german       8295
spanish      3496
japanese     3167
italian   

---
## 2. Fix `author` and `narrator` Columns

**Issue:** Both columns contain prefixes like `Writtenby:` and `Narratedby:` with no spaces between words.  
**Fix:** Strip the prefix, then insert spaces before uppercase letters to separate concatenated names.

In [4]:
# Preview raw values
print('Raw author samples:')
print(df['author'].head(5).tolist())
print('\nRaw narrator samples:')
print(df['narrator'].head(5).tolist())

Raw author samples:
['Writtenby:GeronimoStilton', 'Writtenby:RickRiordan', 'Writtenby:JeffKinney', 'Writtenby:RickRiordan', 'Writtenby:RickRiordan']

Raw narrator samples:
['Narratedby:BillLobely', 'Narratedby:RobbieDaymond', 'Narratedby:DanRussell', 'Narratedby:SoneelaNankani', 'Narratedby:JesseBernstein']


In [5]:
def fix_name(s):
    """Strip prefix and add spaces before uppercase letters."""
    s = str(s)
    s = re.sub(r'(?i)writtenby:', '', s)
    s = re.sub(r'(?i)narratedby:', '', s)
    s = re.sub(r'(\w)([A-Z])', r'\1 \2', s)
    return s.strip()

df['author']   = df['author'].apply(fix_name)
df['narrator'] = df['narrator'].apply(fix_name)

print('Cleaned author samples:')
print(df['author'].head(5).tolist())
print('\nCleaned narrator samples:')
print(df['narrator'].head(5).tolist())

Cleaned author samples:
['Geronimo Stilton', 'Rick Riordan', 'Jeff Kinney', 'Rick Riordan', 'Rick Riordan']

Cleaned narrator samples:
['Bill Lobely', 'Robbie Daymond', 'Dan Russell', 'Soneela Nankani', 'Jesse Bernstein']


---
## 3. Fix `price` Column

**Issue:** `price` is stored as a string — contains `'Free'`, commas, and currency symbols.  
**Fix:** Replace `'Free'` with `0.0`, strip non-numeric characters, cast to float.

In [6]:
print('Unique price patterns (sample):')
print(df['price'].value_counts().head(10))

Unique price patterns (sample):
price
586.00      5533
668.00      4262
703.00      3588
836.00      2704
820.00      2458
233.00      2428
469.00      1703
1,172.00    1683
500.00      1599
501.00      1591
Name: count, dtype: int64


In [7]:
df['price'] = (
    df['price']
    .str.replace(',', '', regex=False)
    .str.replace('$', '', regex=False)
    .str.strip()
    .replace('Free', '0.0')
)
df['price'] = pd.to_numeric(df['price'], errors='coerce')

print(f'Price dtype: {df["price"].dtype}')
print(f'Null prices: {df["price"].isnull().sum()}')
print(df['price'].describe())

Price dtype: float64
Null prices: 0
count    87489.000000
mean       559.009246
std        336.096642
min          0.000000
25%        268.000000
50%        585.000000
75%        755.000000
max       7198.000000
Name: price, dtype: float64


---
## 4. Parse `time` → `duration_minutes`

**Issue:** Duration is stored as text e.g. `'9 hrs and 23 mins'`, `'1 hr'`, `'45 mins'`.  
**Fix:** Use regex to extract hours and minutes, convert to total minutes as integer.

In [8]:
print('Unique time patterns (sample):')
time_patterns = df['time'].str.replace(r'[0-9]', '', regex=True)
print(time_patterns.value_counts().head(10))

Unique time patterns (sample):
time
 hrs and  mins       65090
 mins                12999
 hr and  mins         6347
 hrs                  1158
 hrs and  min         1108
 min                   346
 hr and  min           195
 hr                    185
Less than  minute       61
Name: count, dtype: int64


In [9]:
def parse_duration(t):
    """Convert duration string to total minutes."""
    t = str(t)
    hrs  = re.search(r'(\d+)\s*hr', t)
    mins = re.search(r'(\d+)\s*min', t)
    h = int(hrs.group(1))  if hrs  else 0
    m = int(mins.group(1)) if mins else 0
    return h * 60 + m

df['duration_minutes'] = df['time'].apply(parse_duration)

# Drop original time column — no longer needed
df.drop(columns=['time'], inplace=True)

print(df['duration_minutes'].describe())
print(f'\nZero-duration entries: {(df["duration_minutes"] == 0).sum()}')

count    87489.000000
mean       417.497663
std        364.559399
min          1.000000
25%        142.000000
50%        386.000000
75%        584.000000
max       8595.000000
Name: duration_minutes, dtype: float64

Zero-duration entries: 0


In [10]:
# Replace zero-duration entries with NaN (likely bad data)
df['duration_minutes'] = df['duration_minutes'].replace(0, np.nan)
print(f'Null duration_minutes after fix: {df["duration_minutes"].isnull().sum()}')

Null duration_minutes after fix: 0


---
## 5. Split `stars` → `star_score` + `num_ratings`

**Issue:** The `stars` column combines two pieces of information in one string,  
e.g. `'4.5 out of 5 stars 1,234 ratings'`. Some entries are `'Not rated yet'`.  
**Fix:** Use regex to extract the numeric score and rating count into separate columns.

In [11]:
print('Stars column samples:')
print(df['stars'].head(10).tolist())
print(f'\n"Not rated yet" count: {(df["stars"] == "Not rated yet").sum()}')

Stars column samples:
['5 out of 5 stars34 ratings', '4.5 out of 5 stars41 ratings', '4.5 out of 5 stars38 ratings', '4.5 out of 5 stars12 ratings', '4.5 out of 5 stars181 ratings', '5 out of 5 stars72 ratings', '5 out of 5 stars11 ratings', '5 out of 5 stars50 ratings', '5 out of 5 stars5 ratings', '5 out of 5 stars58 ratings']

"Not rated yet" count: 72417


In [12]:
# Replace unrated entries with NaN before extraction
df['stars'] = df['stars'].replace('Not rated yet', np.nan)

# Extract star score (e.g. 4.5)
df['star_score'] = df['stars'].str.extract(r'^([\d.]+)').astype(float)

# Extract number of ratings (e.g. 1234 from '1,234 ratings')
df['num_ratings'] = (
    df['stars']
    .str.replace(',', '', regex=False)
    .str.extract(r'([\d]+)\s+rating')
    .astype(float)
)

# Fill NaN with 0 for both
df['star_score']  = df['star_score'].fillna(0)
df['num_ratings'] = df['num_ratings'].fillna(0)

# Drop original stars column
df.drop(columns=['stars'], inplace=True)

print('star_score summary:')
print(df['star_score'].describe())
print('\nnum_ratings summary:')
print(df['num_ratings'].describe())

star_score summary:
count    87489.000000
mean         0.767811
std          1.709640
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max          5.000000
Name: star_score, dtype: float64

num_ratings summary:
count    87489.000000
mean         3.723371
std         86.499601
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max      12573.000000
Name: num_ratings, dtype: float64


---
## 6. Parse `releasedate` → `release_year`

**Issue:** `releasedate` is a raw string.  
**Fix:** Parse to datetime and extract the year as an integer for modeling.

In [13]:
print('Sample release dates:')
print(df['releasedate'].head(10).tolist())

Sample release dates:
['04-08-08', '01-05-18', '06-11-20', '05-10-21', '13-01-10', '30-10-18', '25-11-14', '02-05-17', '02-05-17', '24-09-19']


In [14]:
df['releasedate']  = pd.to_datetime(df['releasedate'], errors='coerce')
df['release_year'] = df['releasedate'].dt.year.astype('Int64')

print(df['release_year'].value_counts().sort_index())

release_year
1998        3
1999      233
2000      189
2001      108
2002       75
2003      155
2004      285
2005      363
2006      661
2007      626
2008     1058
2009     1536
2010     1384
2011     1533
2012     1914
2013     3201
2014     2637
2015     3672
2016     3363
2017     4583
2018     6837
2019     8286
2020    10839
2021    23541
2022    10387
2023       14
2024        4
2025        2
Name: count, dtype: Int64


---
## 7. Fix `language` Formatting

**Issue:** Language values may have inconsistent capitalisation e.g. `'english'`, `'ENGLISH'`.  
**Fix:** Capitalize first letter only.

In [15]:
df['language'] = df['language'].str.capitalize()

print('Language value counts:')
print(df['language'].value_counts().head(15))

Language value counts:
language
English       61884
German         8295
Spanish        3496
Japanese       3167
Italian        2694
French         2386
Russian        1804
Danish          935
Portuguese      526
Swedish         515
Hindi           436
Polish          224
Finnish         197
Dutch           190
Tamil           161
Name: count, dtype: int64


---
## 8. Remove Duplicates

**Issue:** Some titles appear multiple times with different release dates.  
**Fix:** Deduplicate on meaningful columns, keeping the most recent entry.

In [16]:
subset_cols = ['name', 'author', 'narrator', 'duration_minutes', 'price']
dupes = df.duplicated(subset=subset_cols).sum()
print(f'Duplicate rows found: {dupes}')

Duplicate rows found: 70


In [17]:
df.sort_values('releasedate', inplace=True)
df.drop_duplicates(subset=subset_cols, keep='last', inplace=True)
df.reset_index(drop=True, inplace=True)

print(f'Rows after deduplication: {len(df)}')

Rows after deduplication: 87419


---
## 9. Final Check & Export

In [18]:
print('=== Final Dataset Shape ===')
print(df.shape)

print('\n=== Column Dtypes ===')
print(df.dtypes)

print('\n=== Null Counts ===')
print(df.isnull().sum())

=== Final Dataset Shape ===
(87419, 10)

=== Column Dtypes ===
name                           str
author                         str
narrator                       str
releasedate         datetime64[us]
language                       str
price                      float64
duration_minutes             int64
star_score                 float64
num_ratings                float64
release_year                 Int64
dtype: object

=== Null Counts ===
name                0
author              0
narrator            0
releasedate         0
language            0
price               0
duration_minutes    0
star_score          0
num_ratings         0
release_year        0
dtype: int64


In [19]:
# Reorder columns cleanly
col_order = ['name', 'author', 'narrator', 'language',
             'duration_minutes', 'releasedate', 'release_year',
             'star_score', 'num_ratings', 'price']
df = df[col_order]
df.head()

,name,author,narrator,language,duration_minutes,releasedate,release_year,star_score,num_ratings,price
0,"Surely You're Joking, Mr. Feynman!",Richard P.Feynman,Raymond Todd,English,691,1998-12-27,1998,4.5,328.0,773.0
1,"Men Are from Mars, Women Are from Venus",John Gray,John Gray,English,568,1998-12-27,1998,4.5,166.0,1519.0
2,Letters to a Young Poet,"Rainer Maria Rilke,Stephen Mitchell-translator",Stephen Mitchell,English,78,1998-12-27,1998,5.0,2.0,233.0
3,"Rhetoric, Poetics and Logic",Aristotle,Frederick Davidson,English,808,1999-05-10,1999,0.0,0.0,773.0
4,Wealth of Nations,Adam Smith,Michael Edwards,English,2119,1999-05-10,1999,0.0,0.0,1338.0


In [20]:
print('=== Summary Statistics (Numeric Columns) ===')
df[['price', 'star_score', 'num_ratings', 'duration_minutes', 'release_year']].describe().round(2)

=== Summary Statistics (Numeric Columns) ===


,price,star_score,num_ratings,duration_minutes,release_year
count,87419.00,87419.00,87419.00,87419.00,87419.0
mean,558.99,0.77,3.37,417.55,2017.99
std,336.06,1.71,71.69,364.62,4.26
min,0.00,0.00,0.00,1.00,1998.0
25%,268.00,0.00,0.00,142.00,2016.0
50%,585.00,0.00,0.00,386.00,2020.0
75%,755.00,0.00,0.00,584.00,2021.0
max,7198.00,5.00,12573.00,8595.00,2025.0


In [21]:
df.to_csv('../data/audible_cleaned.csv', index=False)
print('audible_cleaned.csv saved to ../data/')

audible_cleaned.csv saved to ../data/
